# End of week 1 exercise

Build a tool that uses a local Llama model in Ollama to take a technical question,  
and respond with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports

import json
import shutil
import subprocess
import time
import urllib.request
from urllib.error import URLError

from IPython.display import Markdown, display


In [ ]:
# constants

MODEL_LLAMA = 'llama3.2:1b'

In [ ]:
# set up environment

OLLAMA_URL = "http://localhost:11434/v1"

system_prompt = (
    "You are a helpful technical tutor. Explain code clearly, "
    "focus on what it does and why it works, and keep the answer concise."
)

def messages_for(question):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]

def ollama_is_running():
    try:
        with urllib.request.urlopen("http://localhost:11434/v1", timeout=2) as response:
            response.read(1)
        return True
    except URLError:
        return False

def ensure_ollama_running():
    if ollama_is_running():
        return

    if shutil.which("ollama") is None:
        raise RuntimeError("Ollama is not installed or not on PATH. Start Ollama and try again.")

    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    for _ in range(30):
        if ollama_is_running():
            return
        time.sleep(1)

    raise RuntimeError("Ollama server did not start. Run `ollama serve` and try again.")

def ask_llama(question):
    ensure_ollama_running()

    payload = {
        "model": MODEL_LLAMA,
        "messages": messages_for(question),
        "stream": True,
    }
    request = urllib.request.Request(
        OLLAMA_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request) as response:
        data = json.loads(response.read().decode("utf-8"))
    return data["message"]["content"]

In [ ]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
# Get Llama 3.2 to answer

llama_answer = ask_llama(question)
display(Markdown(llama_answer))